In [1]:
# ============================================================
# 2. Cleaning/01_cleaning_export.ipynb
# Purpose:
# - Read raw data from ../1. Data/weatherAUS.csv
# - Apply the agreed "loose interpolation" + feature engineering
# - Export two datasets:
#   (1) EDA-ready  -> ../1. Data/processed/weatherAUS_eda_ready.csv
#   (2) Model-ready-> ../1. Data/processed/weatherAUS_model_ready.csv
# ============================================================

In [2]:
# Imports 
import warnings
from pathlib import Path
import pandas as pd

In [3]:
# Global settings
warnings.simplefilter("ignore", category=FutureWarning)
warnings.simplefilter("ignore", category=UserWarning)
pd.set_option("display.max_columns", None)

In [4]:
# Paths (relative to this notebook's folder: "2. Cleaning")
# If this notebook lives in: 2. Cleaning/
# then project root is:      ../
project_root = Path("..")
data_dir = project_root / "1. Data"
raw_path = data_dir / "weatherAUS.csv"
processed_dir = data_dir / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)
eda_out_path = processed_dir / "weatherAUS_eda_ready.csv"
model_out_path = processed_dir / "weatherAUS_model_ready.csv"
print("RAW path:", raw_path)
print("EDA export:", eda_out_path)
print("MODEL export:", model_out_path)

RAW path: ..\1. Data\weatherAUS.csv
EDA export: ..\1. Data\processed\weatherAUS_eda_ready.csv
MODEL export: ..\1. Data\processed\weatherAUS_model_ready.csv


In [5]:
# Load raw data
df = pd.read_csv(raw_path, low_memory=False)
print("Raw shape:", df.shape)

Raw shape: (145460, 23)


In [6]:
# Basic prep: Date + sort for time-aware interpolation per Location
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values(["Location", "Date"]).reset_index(drop=True)

In [7]:
# Interpolation to retain rows
# IMPORTANT: This is done on the whole dataframe after sorting by Location+Date.
numeric_cols_to_fill = [
    "WindGustSpeed",
    "Humidity9am", "Humidity3pm",
    "Pressure9am", "Pressure3pm",
    "Cloud9am", "Cloud3pm",
    "Temp9am", "Temp3pm",
    "Evaporation", "Sunshine",
    "WindSpeed9am", "WindSpeed3pm"]

# Interpolate linearly; fill both forward and backward directions
df[numeric_cols_to_fill] = df[numeric_cols_to_fill].interpolate(method="linear", limit_direction="both")

In [8]:
# Feature engineering (Month + daily averages)
month_map = {
    1: "Jan", 2: "Feb", 3: "Mar", 4: "Apr",
    5: "May", 6: "Jun", 7: "Jul", 8: "Aug",
    9: "Sep", 10: "Oct", 11: "Nov", 12: "Dec"}
month_order = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]

# Create Month label
df["Month"] = df["Date"].dt.month.map(month_map)
# Make Month ordered categorical (useful for plots downstream)
df["Month"] = pd.Categorical(df["Month"], categories=month_order, ordered=True)

# Daily averages from 9am/3pm
df["WindSpeed"] = df[["WindSpeed9am", "WindSpeed3pm"]].mean(axis=1)
df["Humidity"] = df[["Humidity9am", "Humidity3pm"]].mean(axis=1)
df["Pressure"] = df[["Pressure9am", "Pressure3pm"]].mean(axis=1)
df["Cloud"] = df[["Cloud9am", "Cloud3pm"]].mean(axis=1)
df["Temperature"] = df[["Temp9am", "Temp3pm"]].mean(axis=1)

In [9]:
# (A) EDA-ready dataset
# - Keeps Location / RainToday / Rainfall for EDA plots
# - Keeps target as original ("Yes"/"No") plus a binary helper
df_eda_ready = df.dropna(subset=["RainTomorrow"]).copy()

# Binary helper for rates in EDA (Yes=1 means rain tomorrow)
df_eda_ready["RainTomorrow_bin"] = (df_eda_ready["RainTomorrow"].map({"No": 0, "Yes": 1}).astype(int))
print("\nEDA-ready shape:", df_eda_ready.shape)
print("EDA-ready target distribution (RainTomorrow_bin):")
print(df_eda_ready["RainTomorrow_bin"].value_counts(normalize=True))

# Export EDA-ready
df_eda_ready.to_csv(eda_out_path, index=False)


EDA-ready shape: (142193, 30)
EDA-ready target distribution (RainTomorrow_bin):
RainTomorrow_bin
0    0.775819
1    0.224181
Name: proportion, dtype: float64


In [10]:
# (B) Model-ready dataset
# - Drops leakage/unwanted columns (RainToday/Rainfall/Location/etc.)
# - Drops rows with missing required features
# - Target becomes 0/1
# - NO OHE here (OHE happens in modeling notebook)
cols_to_drop = [
    "Rainfall", "RainToday",          # EDA-only / leakage-ish
    "Location",                       # drop for general model
    "MinTemp", "MaxTemp",             # Temperature average used instead
    "WindDir9am", "WindDir3pm",       # WindGustDir used instead
    "Date",                           # not used (Month kept)
    "WindSpeed9am", "WindSpeed3pm",   # replaced by WindSpeed avg
    "Humidity9am", "Humidity3pm",     # replaced by Humidity avg
    "Pressure9am", "Pressure3pm",     # replaced by Pressure avg
    "Cloud9am", "Cloud3pm",           # replaced by Cloud avg
    "Temp9am", "Temp3pm"]             # replaced by Temperature avg

required_features = [
    "Evaporation", "Sunshine", "WindGustSpeed",
    "WindSpeed", "Humidity", "Pressure", "Cloud", "Temperature",
    "WindGustDir", "Month"]

# Drop unwanted columns
df_model_ready = df.drop(columns=cols_to_drop, errors="ignore").copy()

# Keep only rows where target exists
df_model_ready = df_model_ready.dropna(subset=["RainTomorrow"]).reset_index(drop=True)

# Drop rows where required features are missing
df_model_ready = df_model_ready.dropna(subset=required_features).reset_index(drop=True)

# Map target to 0/1
df_model_ready["RainTomorrow"] = df_model_ready["RainTomorrow"].map({"No": 0, "Yes": 1}).astype(int)

# Keep ONLY features + target (clean contract between notebooks)
df_model_ready = df_model_ready[required_features + ["RainTomorrow"]].copy()

print("\nModel-ready shape:", df_model_ready.shape)
print("Model-ready target distribution:")
print(df_model_ready["RainTomorrow"].value_counts(normalize=True))

# Export Model-ready
df_model_ready.to_csv(model_out_path, index=False)

print("\nDONE: Exports created successfully.")


Model-ready shape: (132863, 11)
Model-ready target distribution:
RainTomorrow
0    0.778644
1    0.221356
Name: proportion, dtype: float64

DONE: Exports created successfully.
